# EDEN-02 — RGB-D acquisition and pairing | v1.1
**Phase 1: Data engineering · Notebook 02 of 15 · CPU + Internet**

EDEN-01 → **EDEN-02** → EDEN-03 (segmentation and depth geometry). We obtain a small, real, synchronized color/depth subset across six greenhouse compartments; we do not train a model or reconstruct 3D yet. The official canopy ZIP is 31.4 GB: never download it wholesale to Kaggle. A pilot is for engineering, not final research accuracy. Preserve native image bytes and split labels.

**Input:** attach the successful EDEN-01 output (`results_1.1.zip` or `EDEN-01-Climate-ETL-Results-v1.1.zip`), either as saved notebook output or as an extracted Kaggle dataset. This notebook discovers the `EDEN-01` manifest, verifies hashes and reads its split definitions. No EDEN-00 input required. Set Internet On, accelerator None. If using an authorized local canopy ZIP or extracted camera directory, internet is unnecessary.

**Outputs:** `eden/vision/images.parquet`, `images.csv`, `rgb/`, `depth/`, `calibration/`, `reports/`, `data/splits.json`, `manifest.json` and `/kaggle/working/EDEN-02-RGBD-Acquisition-Results-v1.1.zip`. Keep EDEN-01 separately attached for later climate fusion. The ZIP contains only EDEN-02 artifacts, not a duplicate of climate data.

**Run cells in order.** If HTTP ranges or automatic filename pairing fail, the notebook stops with actionable diagnostics; it never silently substitutes fabricated images or downloads 31 GB. A manual mapping option is provided after inventory. Revision 1.1: Kaggle reached Cell 6 and revealed six camera-specific JSON files under dataset4tu/configs/. This revision uses those filenames, not the generic filename in the README. Image decoding and numeric calibration still require live execution.

## Cell 1 — Environment and pilot settings

Paths identify files; configuration constants define resource budgets. `Path` joins paths safely. A small pilot of up to 36 pairs (2 per compartment per chronological split) covers growth stages without exhausting storage. Both image modalities stay at original resolution. Only previews are downsampled. Set `UPSTREAM` only if multiple EDEN-01 versions are found. `CAMERA_SOURCE` optionally points to an authorized complete ZIP or extracted directory. `PAIR_OVERRIDE` optionally points to a CSV with exact `rgb_member,depth_member,compartment_id,local_time` columns. `CALIBRATION_OVERRIDE` maps camera IDs to exact JSON member names when automatic association is ambiguous.

In [ ]:
from pathlib import Path
import io, os, re, json, hashlib, zipfile, shutil, math
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import requests
import cv2
import importlib.util
assert importlib.util.find_spec('pyarrow') is not None, 'PyArrow is required for Parquet; install it before running the notebook.'
import matplotlib.pyplot as plt
from IPython.display import display
from collections import defaultdict

INPUT = Path(os.environ.get('EDEN_INPUT', '/kaggle/input'))
WORK = Path(os.environ.get('EDEN_WORK', '/kaggle/working'))
ROOT = WORK / 'eden'
VISION = ROOT / 'vision'
REPORTS = ROOT / 'reports'
UPSTREAM = None  # e.g. Path('/kaggle/input/.../results_1.1.zip')
CAMERA_SOURCE = None  # authorized local canopy ZIP or extracted camera root (optional)
PAIR_OVERRIDE = None  # optional reviewed pair CSV
CALIBRATION_OVERRIDE = {}  # camera ID -> exact archive member name
PAIRS_PER_STRATUM = 2
MAX_SELECTED_BYTES = 1_500_000_000
MAX_NETWORK_BYTES = 2_000_000_000
MAX_MEMBER_BYTES = 100_000_000
DOI = '10.4121/fa102772-32db-4b30-bace-12f2016722ce.v1'
CAMERAS = {'3.01': ('tomatonuts','camir_27'), '3.02': ('mugrow','cam_21'),
           '3.03': ('agrifusion','cam_20'), '3.06': ('reference','cam_1'),
           '3.07': ('ideas','cam_19'), '3.08': ('trigger','cam_18')}
for p in [VISION/'rgb',VISION/'depth',VISION/'calibration',REPORTS,ROOT/'data']:
    p.mkdir(parents=True,exist_ok=True)
assert shutil.disk_usage(WORK).free > 2*MAX_SELECTED_BYTES + 100_000_000, 'Insufficient space for pilot and export ZIP'
def save_json(path,obj):
    path.write_text(json.dumps(obj,indent=2,ensure_ascii=False),encoding='utf-8')
def sha(b): return hashlib.sha256(b).hexdigest()
def file_sha(path):
    h=hashlib.sha256()
    with path.open('rb') as f:
        for b in iter(lambda:f.read(1024*1024),b''): h.update(b)
    return h.hexdigest()
print('OpenCV',cv2.__version__,'Pandas',pd.__version__,'Free GiB',round(shutil.disk_usage(WORK).free/2**30,2))

## Cell 2 — Find and verify EDEN-01

The manifest is a receipt listing expected files, sizes and SHA-256 hashes. This cell accepts an extracted dataset or a ZIP (including your `results_1.1.zip`), locates the actual producer, and verifies all outputs. It refuses ambiguous inputs or a failed ETL. The climate table provides actual compartment coverage; split definitions are inherited unchanged. Pandas reads Parquet using PyArrow. If this dependency is missing, enable Internet and install `pyarrow` in a separate cell.

In [ ]:
def manifest_candidates():
    paths = [Path(UPSTREAM)] if UPSTREAM else sorted(INPUT.rglob('manifest.json')) + sorted(INPUT.rglob('*.zip'))
    found=[]
    for p in paths:
        if p.is_dir(): p=p/'manifest.json'
        if p.name=='manifest.json':
            try:
                b=p.read_bytes(); m=json.loads(b)
                if m.get('producer')=='EDEN-01': found.append((p,None,b,m))
            except (OSError,ValueError): continue
        elif p.suffix.lower()=='.zip':
            try:
                with zipfile.ZipFile(p) as z:
                    for name in z.namelist():
                        if Path(name).name!='manifest.json' or '__MACOSX' in Path(name).parts: continue
                        b=z.read(name); m=json.loads(b)
                        if m.get('producer')=='EDEN-01': found.append((p,name,b,m))
            except (OSError,ValueError,zipfile.BadZipFile): continue
    return found
found=manifest_candidates()
assert len(found)==1, f'Attach exactly one EDEN-01 output or set UPSTREAM. Found: {[str(x[0]) for x in found]}'
up_path,up_member,up_bytes,up_manifest=found[0]
assert up_manifest.get('status')=='PASS_ETL', 'EDEN-01 must pass its acceptance gate'
assert up_manifest.get('source_doi')==DOI, 'Unexpected source version'
def upstream(name):
    if up_member is None: return (up_path.parent/name).read_bytes()
    with zipfile.ZipFile(up_path) as z:
        return z.read(str(Path(up_member).parent/name))
for entry in up_manifest['files']:
    payload=upstream(entry['path'])
    assert len(payload)==entry['size'] and sha(payload)==entry['sha256'], entry['path']
SPLITS=json.loads(upstream('data/splits.json'))
SOURCE=json.loads(upstream('source_manifest.json'))
climate=pd.read_parquet(io.BytesIO(upstream('data/climate.parquet')),columns=['experiment_id','compartment_id','ts_utc','split','chronological_split'])
climate['ts_utc']=pd.to_datetime(climate.ts_utc,utc=True)
assert set(climate.compartment_id)==set(CAMERAS)
assert climate.experiment_id.nunique()==1 and climate.experiment_id.iloc[0]==SPLITS['experiment_id']
PRIMARY={c:s for s,ids in SPLITS['primary']['compartments'].items() for c in ids}
assert set(PRIMARY)==set(CAMERAS) and climate.split.eq(climate.compartment_id.map(PRIMARY)).all()
BOUNDS=climate.groupby('compartment_id').ts_utc.agg(['min','max']).to_dict('index')
(ROOT/'data'/'splits.json').write_bytes(upstream('data/splits.json'))
(ROOT/'source_manifest.json').write_bytes(upstream('source_manifest.json'))
print('PASS: EDEN-01 manifest and',len(up_manifest['files']),'file hashes verified')
print('Input:',up_path,'| climate rows:',len(climate))
display(pd.DataFrame(BOUNDS).T)

## Cell 3 — Open the official canopy archive safely

HTTP Range requests ask the server for specific bytes. ZIP central-directory metadata sits near the end, so Python can enumerate members without fetching the full archive. The seekable reader below checks HTTP 206, exact Content-Range, publisher archive size, response encoding and a total transfer budget. Each selected ZIP member is CRC-checked by `zipfile` when read completely. A locally attached complete ZIP is checked against the publisher MD5; extracted local files have weaker source verification and are explicitly labeled. Remote partial access cannot verify whole-archive MD5. If range access is blocked, stop and attach an authorized ZIP or extracted canopy folder; do not switch to an unbounded download.

In [ ]:
archive_name='autonomous_greenhouse_challenge4_canopy_camera.zip'
spec=SOURCE['files'][archive_name]
assert spec['size']==31_416_872_168 and spec['md5']=='671a1c6e2b2e1627f821a36e431ceb28'
class RangeReader(io.RawIOBase):
    def __init__(self,url,size,budget):
        super().__init__(); self.url=url; self.size=size; self.budget=budget
        self.pos=0; self.transferred=0; self.session=requests.Session(); self.validator=None
        self.fetch(0,1)  # probe one byte; rejects an HTTP 200 without reading its body
    def readable(self): return True
    def seekable(self): return True
    def tell(self): return self.pos
    def seek(self,offset,whence=0):
        new={0:offset,1:self.pos+offset,2:self.size+offset}[whence]
        if new<0: raise ValueError('Negative seek')
        self.pos=new; return new
    def fetch(self,start,n):
        if n==0: return b''
        if self.transferred+n>self.budget: raise RuntimeError('Network budget exceeded')
        end=start+n-1
        headers={'Range':f'bytes={start}-{end}','Accept-Encoding':'identity'}
        if self.validator: headers['If-Range']=self.validator
        with self.session.get(self.url,headers=headers,stream=True,timeout=(20,120)) as r:
            if r.status_code!=206: raise RuntimeError(f'Range unsupported or resource changed (HTTP {r.status_code}). Attach authorized camera data instead.')
            if r.headers.get('Content-Range')!=f'bytes {start}-{end}/{self.size}': raise RuntimeError('Unexpected Content-Range')
            if r.headers.get('Content-Encoding','identity').lower()!='identity': raise RuntimeError('Encoded range response')
            validator=r.headers.get('ETag') or r.headers.get('Last-Modified')
            if self.validator and validator!=self.validator: raise RuntimeError('Remote archive changed during access')
            if validator and not validator.startswith('W/'): self.validator=validator
            self.transferred+=n  # count requested bytes even if body fails
            data=r.raw.read(n+1)
            if len(data)!=n: raise RuntimeError('Incomplete/oversize range response')
            return data
    def read(self,n=-1):
        remaining=max(0,self.size-self.pos)
        n=remaining if n<0 else min(n,remaining)
        b=self.fetch(self.pos,n); self.pos+=len(b); return b
    def close(self):
        if hasattr(self,'session'): self.session.close()
        super().close()

def genuine(name):
    p=Path(name)
    return '__MACOSX' not in p.parts and not any(part.startswith('.') for part in p.parts)
reader=None; z=None; local_root=None
if CAMERA_SOURCE is not None:
    p=Path(CAMERA_SOURCE)
    if p.is_dir():
        local_root=p; source_mode='attached_extracted_unverified_archive'
        members={str(f.relative_to(p)):f.stat().st_size for f in p.rglob('*') if f.is_file() and genuine(str(f.relative_to(p)))}
    else:
        h=hashlib.md5()
        with p.open('rb') as f:
            for b in iter(lambda:f.read(1024*1024),b''): h.update(b)
        assert p.stat().st_size==spec['size'] and h.hexdigest()==spec['md5']
        z=zipfile.ZipFile(p); source_mode='local_zip_md5_verified'
else:
    reader=RangeReader(spec['url'],spec['size'],MAX_NETWORK_BYTES)
    z=zipfile.ZipFile(reader); source_mode='http_range_partial_crc'
if z is not None:
    infos=[x for x in z.infolist() if not x.is_dir() and genuine(x.filename)]
    assert len({x.filename for x in infos})==len(infos),'Duplicate archive names'
    members={x.filename:x.file_size for x in infos}
assert members,'No genuine archive members'
def read_member(name):
    assert name in members and 0<members[name]<=MAX_MEMBER_BYTES,name
    b=z.read(name) if z is not None else (local_root/name).read_bytes()
    assert len(b)==members[name],name
    return b
pd.DataFrame([{'member':k,'size_bytes':v} for k,v in members.items()]).to_csv(REPORTS/'camera_archive_inventory.csv',index=False)
print('Mode:',source_mode,'| Genuine members:',len(members))
print('\n'.join(list(members)[:40]))

## Cell 4 — Detect RGB/depth pairs from actual filenames

The publisher describes six fixed cameras and timestamps in `YYYY_MM_DD_hh_mm_ss` local Dutch time. Pairing requires the same compartment and exact capture timestamp. We infer modality only from explicit path tokens (RGB/color/depth), not image ordering; ambiguous, duplicate or unmatched paths are reported. The manual mapping CSV provides an escape hatch when the actual archive naming differs. No approximate pairing is allowed. DST ambiguous/nonexistent times are rejected rather than guessed. Camera IDs are mapped from the verified EDEN-00 README.

In [ ]:
IMAGE_EXT={'.png','.jpg','.jpeg','.tif','.tiff'}
TS=re.compile(r'(?<!\d)(\d{4}[_-]\d{2}[_-]\d{2})[T_ -](\d{2}[_:-]\d{2}[_:-]\d{2})(?!\d)')
def identify(name):
    low=name.lower(); hits=[]
    for cid,(team,camera) in CAMERAS.items():
        if any(re.search(r'(?<![a-z0-9])'+re.escape(t)+r'(?![a-z0-9])',low) for t in (team,camera,cid)):
            hits.append(cid)
    t=TS.search(low)
    # Use the filename first; directory tokens are fallback.
    tokens=set(re.split(r'[^a-z0-9]+',Path(low).stem))
    modality_tokens={'depth','depthmap','depthimage','depthaligned','rgb','color','colour','rgbimage','colorimage'}
    if not tokens & modality_tokens:
        tokens=set(re.split(r'[^a-z0-9]+',str(Path(low).parent)))
    depth=bool(tokens & {'depth','depthmap','depthimage','depthaligned'})
    rgb=bool(tokens & {'rgb','color','colour','rgbimage','colorimage'})
    # Depth directory can be nested below an RGB-D root; reject ambiguous explicit modalities.
    if len(hits)!=1 or not t or depth==rgb: return None
    date=t.group(1).replace('_','-'); clock=t.group(2).replace('_',':').replace('-',':')
    return hits[0],date+' '+clock,'depth' if depth else 'rgb'
groups=defaultdict(lambda: defaultdict(list)); unresolved=[]
for name in members:
    if Path(name).suffix.lower() not in IMAGE_EXT: continue
    key=identify(name)
    if key is None: unresolved.append(name); continue
    cid,stamp,kind=key; groups[(cid,stamp)][kind].append(name)
pairs=[]; ambiguous=[]
for (cid,stamp),g in sorted(groups.items()):
    if len(g['rgb'])==len(g['depth'])==1:
        pairs.append(dict(compartment_id=cid,local_time=stamp,rgb_member=g['rgb'][0],depth_member=g['depth'][0]))
    else: ambiguous.append(dict(compartment_id=cid,local_time=stamp,rgb=g['rgb'],depth=g['depth']))
save_json(REPORTS/'pairing_diagnostics.json',dict(unresolved=unresolved,ambiguous=ambiguous))
if PAIR_OVERRIDE is not None:
    pairs=pd.read_csv(PAIR_OVERRIDE,dtype={'compartment_id':str}).to_dict('records')
assert pairs, 'No unambiguous pairs. Inspect camera_archive_inventory.csv and pairing_diagnostics.json; provide PAIR_OVERRIDE with exact names.'
paired=pd.DataFrame(pairs)
assert set(paired.compartment_id)<=set(CAMERAS)
assert paired.rgb_member.ne(paired.depth_member).all()
assert paired.rgb_member.is_unique and paired.depth_member.is_unique
assert not set(paired.rgb_member)&set(paired.depth_member)
assert not paired.duplicated(['compartment_id','local_time']).any()
for name in [*paired.rgb_member,*paired.depth_member]: assert name in members and Path(name).suffix.lower() in IMAGE_EXT,name
paired['ts_utc']=pd.to_datetime(paired.local_time,errors='raise').dt.tz_localize('Europe/Amsterdam',ambiguous='NaT',nonexistent='NaT').dt.tz_convert('UTC')
paired['split']=paired.compartment_id.map(PRIMARY)
chron=SPLITS['chronological']
cut1=pd.Timestamp(chron['train_end_exclusive']); vstart=pd.Timestamp(chron['validation_start_inclusive'])
cut2=pd.Timestamp(chron['validation_end_exclusive']); tstart=pd.Timestamp(chron['test_start_inclusive'])
t=paired.ts_utc
paired['chronological_split']=np.select([t<cut1,(t>=vstart)&(t<cut2),t>=tstart],['train','validation','test'],default='embargo')
paired['in_climate_coverage']=[pd.notna(t) and BOUNDS[c]['min']<=t<=BOUNDS[c]['max'] for c,t in zip(paired.compartment_id,paired.ts_utc)]
paired.to_csv(REPORTS/'all_detected_pairs.csv',index=False)
eligible=paired[paired.in_climate_coverage & paired.chronological_split.ne('embargo')].copy()
assert set(eligible.compartment_id)==set(CAMERAS), 'Not all six compartments have eligible pairs. Review diagnostics/override.'
print('Pairs:',len(paired),'| Eligible:',len(eligible),'| Unresolved image names:',len(unresolved),'| Ambiguous keys:',len(ambiguous))
display(eligible.groupby(['compartment_id','chronological_split']).size().unstack(fill_value=0))

## Cell 5 — Select a small, reproducible pilot

Stratified sampling selects across compartment and chronological split. Within each group we pick evenly spaced dates/times, not by image appearance or outcomes. Both evaluation protocols are inherited; embargo images are excluded. We record coverage and enforce all six compartments and all three chronological splits globally. This is an engineering pilot, not a representative final dataset. Change `PAIRS_PER_STRATUM` only after the pilot works.

In [ ]:
parts=[]
for _,g in eligible.groupby(['compartment_id','chronological_split'],sort=True):
    g=g.sort_values('ts_utc'); k=min(PAIRS_PER_STRATUM,len(g))
    idx=np.linspace(0,len(g)-1,k).round().astype(int) if k>1 else [len(g)//2]
    parts.append(g.iloc[idx])
selected=pd.concat(parts,ignore_index=True).sort_values(['compartment_id','ts_utc']).reset_index(drop=True)
assert set(selected.split)=={'train','validation','test'} and set(selected.chronological_split)=={'train','validation','test'}
selected['image_id']=[sha(f"{c}|{t.isoformat()}".encode())[:20] for c,t in zip(selected.compartment_id,selected.ts_utc)]
assert selected.image_id.is_unique
assert ((selected.split=='train') & (selected.chronological_split=='train')).any(), 'No training-only pairs for montage; inspect split coverage'
assert all(0 < members[name] <= MAX_MEMBER_BYTES for name in [*selected.rgb_member, *selected.depth_member]), 'Selected member exceeds size limit; review inventory'
selected['pair_bytes']=[members[r]+members[d] for r,d in zip(selected.rgb_member,selected.depth_member)]
assert selected.pair_bytes.sum()<=MAX_SELECTED_BYTES,'Pilot exceeds storage budget; reduce PAIRS_PER_STRATUM'
selected.to_csv(REPORTS/'selected_pairs.csv',index=False)
print('Selected pairs:',len(selected),'| native bytes:',int(selected.pair_bytes.sum()))
display(selected.groupby(['compartment_id','chronological_split']).size().unstack(fill_value=0))

## Cell 6 — Acquire camera calibration metadata (fixed in v1.1)

The actual archive lists `dataset4tu/configs/camir_27.json`, `cam_21.json`, `cam_20.json`, `cam_1.json`, `cam_19.json` and `cam_18.json`. The README's generic `canopy_camera_config.json` is not present. Match each camera ID to its exact JSON filename (not a substring), preserving the original JSON bytes and SHA-256. `CALIBRATION_OVERRIDE` can specify a reviewed exact archive member for a camera if needed. Print JSON keys and recursively locate candidate intrinsic fields; absence of a recognized key is reported, not replaced with invented values. A camera-specific filename establishes an association, not proof of numerical calibration. EDEN-03 must verify the intrinsics, depth units and reference plane before metric reconstruction. The pilot gate requires all six metadata files and valid associations, not verified metric geometry.

In [ ]:
# Actual archive filenames observed in the Kaggle run; exact IDs avoid cam_1/cam_18 collisions.
expected_cameras = {cam for team, cam in CAMERAS.values()}
assert set(CALIBRATION_OVERRIDE) <= expected_cameras, 'Unknown camera in CALIBRATION_OVERRIDE'
json_files = sorted(name for name in members if Path(name).suffix.lower() == '.json')
print('JSON files found:', len(json_files))
for name in json_files:
    print(name)

# Discover all six files before downloading anything; prefer the documented configs directory.
chosen = {}
for camera in sorted(expected_cameras):
    if camera in CALIBRATION_OVERRIDE:
        hits = [CALIBRATION_OVERRIDE[camera]]
        assert hits[0] in json_files, f'Invalid override for {camera}: {hits[0]}'
    else:
        exact = [name for name in json_files if Path(name).stem.lower() == camera.lower()]
        in_configs = [name for name in exact if 'configs' in [part.lower() for part in Path(name).parts[:-1]]]
        hits = in_configs or exact
    assert len(hits) == 1, f'{camera}: expected one calibration JSON, found {hits}; review inventory or set override'
    chosen[camera] = hits[0]
assert len(set(chosen.values())) == len(expected_cameras), 'A calibration file was assigned to multiple cameras; review mappings'

def intrinsic_candidates(obj, path=''):
    # Record potential keys only; their matrix layout, units and validity remain unverified.
    hits = []
    if isinstance(obj, dict):
        for key, value in obj.items():
            field_path = f'{path}/{key}'
            normalized = re.sub(r'[^a-z0-9]', '', key.lower())
            if normalized in {'colorint', 'colourint', 'rgbint', 'intrinsics', 'cameramatrix', 'intrinsicmatrix', 'depthint'}:
                hits.append(field_path)
            hits.extend(intrinsic_candidates(value, field_path))
    elif isinstance(obj, list):
        for i, value in enumerate(obj):
            if isinstance(value, (dict, list)):
                hits.extend(intrinsic_candidates(value, f'{path}/{i}'))
    return hits

config_records = []
config_by_camera = {}
for camera, name in chosen.items():
    assert 0 < members[name] <= 2_000_000, f'Unexpected metadata size: {name}'
    b = read_member(name)  # Original bytes, using existing range/ZIP reader
    obj = json.loads(b)
    assert isinstance(obj, dict) and obj, f'Expected a nonempty camera configuration object: {name}'
    calibration_id = sha(name.encode())[:16]
    dest = VISION / 'calibration' / f'{calibration_id}.json'
    dest.write_bytes(b)
    paths = intrinsic_candidates(obj)
    record = dict(calibration_id=calibration_id, camera_id=camera, source_member=name,
                  path=str(dest.relative_to(ROOT)), sha256=sha(b), top_level_keys=list(obj),
                  intrinsic_candidate_paths=paths, numeric_calibration_verified=False,
                  association='manual_candidate' if camera in CALIBRATION_OVERRIDE else 'camera_filename_candidate')
    config_records.append(record)
    config_by_camera[camera] = record
    print(camera, '→', name, '| keys:', list(obj), '| intrinsic candidates:', paths or 'not recognized; review in EDEN-03')

# Define the same interface used by Cell 7. No undefined configs variable.
def config_for(cid):
    camera = CAMERAS[cid][1]
    record = config_by_camera[camera]
    return record['calibration_id'], record['association']

assert set(config_by_camera) == expected_cameras
save_json(VISION / 'calibration' / 'index.json', config_records)
print('PASS: calibration metadata preserved for', len(config_records), 'cameras')
for cid in CAMERAS:
    print(cid, CAMERAS[cid][1], config_for(cid))
print('PENDING: numeric intrinsics, depth units and reference-plane verification')

## Cell 7 — Download only selected images and decode

Read selected ZIP members, preserving original image bytes. OpenCV `imdecode` uses unchanged mode to preserve native depth values and bit depth. BGR is converted to RGB only for display. We require a color array with 3/4 channels and an integer/float single-channel depth array with matching dimensions and nonzero finite values. These are engineering checks, not metric depth calibration. Zero is treated as a provisional invalid-depth indicator; the publisher convention still needs verification. Depth images are never normalized or resized on disk. Save SHA-256 for each selected file.

In [ ]:
rows=[]; failures=[]
for r in selected.itertuples(index=False):
    try:
        rb=read_member(r.rgb_member); db=read_member(r.depth_member)
        rgb=cv2.imdecode(np.frombuffer(rb,dtype=np.uint8),cv2.IMREAD_UNCHANGED)
        depth=cv2.imdecode(np.frombuffer(db,dtype=np.uint8),cv2.IMREAD_UNCHANGED)
        assert rgb is not None and rgb.ndim==3 and rgb.shape[2] in (3,4),'Invalid RGB decode'
        assert depth is not None and depth.ndim==2 and depth.dtype.kind in 'uif','Depth must be single-channel numeric'
        assert depth.shape==rgb.shape[:2],'RGB/depth dimensions differ'
        assert depth.dtype.itemsize>=2,'8-bit depth may be visualized rather than native; review source'
        valid=np.isfinite(depth)&(depth>0)
        assert valid.any(),'No positive finite depth'
        rp=VISION/'rgb'/(r.image_id+Path(r.rgb_member).suffix.lower())
        dp=VISION/'depth'/(r.image_id+Path(r.depth_member).suffix.lower())
        rp.write_bytes(rb); dp.write_bytes(db)
        cal,cal_status=config_for(r.compartment_id)
        rows.append(dict(image_id=r.image_id,experiment_id=SPLITS['experiment_id'],compartment_id=r.compartment_id,
            camera_id=CAMERAS[r.compartment_id][1],ts_utc=r.ts_utc,local_time=r.local_time,split=r.split,
            chronological_split=r.chronological_split,rgb_path=str(rp.relative_to(ROOT)),depth_path=str(dp.relative_to(ROOT)),
            rgb_member=r.rgb_member,depth_member=r.depth_member,rgb_sha256=sha(rb),depth_sha256=sha(db),
            width=rgb.shape[1],height=rgb.shape[0],rgb_dtype=str(rgb.dtype),depth_dtype=str(depth.dtype),
            valid_depth_fraction=float(valid.mean()),quality='decode_pair_pass',calibration_id=cal,calibration_status=cal_status,
            depth_unit='unverified_native',source_verification=source_mode))
        print('OK',r.compartment_id,r.local_time,rgb.shape[:2],depth.dtype)
    except Exception as e:
        failures.append(dict(image_id=r.image_id,error=str(e))); print('FAIL',r.image_id,e)
        if isinstance(e,RuntimeError): break  # stop on budget / remote errors
save_json(REPORTS/'decode_failures.json',failures)
assert not failures and len(rows)==len(selected),'Some selected pairs failed; inspect decode_failures.json before continuing'
images=pd.DataFrame(rows)
images.to_parquet(VISION/'images.parquet',index=False,engine='pyarrow')
images.to_csv(VISION/'images.csv',index=False)
print('Acquired:',len(images),'pairs; network bytes:',reader.transferred if reader else 'local')

## Cell 8 — Visual quality check

A montage helps identify mismatched camera views or invalid depth. Only pairs belonging to training in BOTH protocols are previewed; no held-out images are used to tune visual processing. Color is displayed naturally. Depth is shown using display-only percentiles; the original values remain untouched. A human still needs to inspect the montage before approving geometry.

In [ ]:
preview=images[(images.split=='train')&(images.chronological_split=='train')].groupby('compartment_id').head(1)
assert len(preview)>0
fig,axes=plt.subplots(len(preview),2,figsize=(10,3*len(preview)),squeeze=False)
for ax,r in zip(axes,preview.itertuples(index=False)):
    rgb=cv2.imread(str(ROOT/r.rgb_path),cv2.IMREAD_COLOR)
    d=cv2.imread(str(ROOT/r.depth_path),cv2.IMREAD_UNCHANGED)
    rgb=cv2.cvtColor(rgb,cv2.COLOR_BGR2RGB); valid=np.isfinite(d)&(d>0)
    lo,hi=np.percentile(d[valid],[2,98]); hi=max(hi,lo+1)
    ax[0].imshow(cv2.resize(rgb,(640,360),interpolation=cv2.INTER_AREA)); ax[0].set_title(r.compartment_id+' RGB')
    ax[1].imshow(np.ma.masked_where(~valid,d),cmap='viridis',vmin=lo,vmax=hi); ax[1].set_title('Depth (native units, display only)')
    for a in ax: a.axis('off')
fig.tight_layout(); fig.savefig(REPORTS/'rgbd_training_montage.png',dpi=120); plt.show(); plt.close(fig)
display(images.groupby(['compartment_id','split']).valid_depth_fraction.agg(['count','min','mean']))

## Cell 9 — Acceptance gate and named results ZIP

Assertions ensure split integrity, matching images, preserved file hashes and Parquet readback. This is an acquisition pass, not approval of metric geometry or scientific accuracy. Unresolved calibration association blocks a full acquisition pass; if it occurs, inspect the calibration paths and set an explicit override. Manual montage review and depth unit confirmation remain pending. The output manifest links the exact EDEN-01 version and lists all EDEN-02 files. The ZIP is stored outside `eden/` so it cannot include itself. Rerunning with a different selection may leave old files: use a fresh working directory or remove only stale EDEN-02 files after review.

In [ ]:
assert len(images)==len(selected) and images.image_id.is_unique
assert not images.duplicated(['experiment_id','compartment_id','ts_utc']).any()
assert str(images.ts_utc.dt.tz)=='UTC'
assert images.split.eq(images.compartment_id.map(PRIMARY)).all()
assert images.chronological_split.ne('embargo').all()
assert images.calibration_id.notna().all(),'Calibration association unresolved; set CALIBRATION_OVERRIDE and rerun Cells 6–9'
assert len(config_records)==len(CAMERAS) and len({r['calibration_id'] for r in config_records})==len(CAMERAS)
for record in config_records:
    assert file_sha(ROOT/record['path'])==record['sha256'], record['source_member']
for row in images.itertuples(index=False):
    assert row.camera_id==CAMERAS[row.compartment_id][1]
    assert row.calibration_id==config_by_camera[row.camera_id]['calibration_id'], row.image_id

for r in images.itertuples(index=False):
    assert file_sha(ROOT/r.rgb_path)==r.rgb_sha256 and file_sha(ROOT/r.depth_path)==r.depth_sha256
pd.testing.assert_frame_equal(pd.read_parquet(VISION/'images.parquet'),images)
assert (ROOT/'data'/'splits.json').read_bytes()==upstream('data/splits.json')
# Exclude stale files by requiring the image folders to exactly match this selection.
for kind in ('rgb','depth'):
    assert {str(p.relative_to(ROOT)) for p in (VISION/kind).iterdir() if p.is_file()}==set(images[kind+'_path']), 'Stale image files: use a clean working directory'
report=dict(status='PASS_RGBD_PILOT',source_mode=source_mode,source_doi=DOI,selected_pairs=len(images),calibration_files=len(config_records),numeric_calibration_verified=False,
            eligible_pairs=len(eligible),unresolved_names=len(unresolved),ambiguous_keys=len(ambiguous),
            whole_archive_md5_verified=(source_mode=='local_zip_md5_verified'),
            member_crc_checked=(z is not None),network_bytes=reader.transferred if reader else None,
            pending=['Human montage review','Depth units and invalid-code verification','Numeric camera intrinsics and reference plane verification','Expanded research sampling'])
save_json(REPORTS/'acquisition_report.json',report)
(ROOT/'LICENSES.txt').write_text('Maree et al. (2025), 4th Autonomous Greenhouse Challenge: Dwarf Tomato Timeseries and Images, v1. 4TU.ResearchData. DOI: '+DOI+'\nCC BY 4.0: https://creativecommons.org/licenses/by/4.0/\nEDEN selected a pilot; native image bytes preserved. Metadata and previews generated.\n',encoding='utf-8')
files=[dict(path=str(p.relative_to(ROOT)),size=p.stat().st_size,sha256=file_sha(p)) for p in sorted(ROOT.rglob('*')) if p.is_file() and p!=ROOT/'manifest.json']
manifest=dict(schema_version='1.0',producer='EDEN-02',code_version='1.1',status='PASS_RGBD_PILOT',source_doi=DOI,
              upstream_manifest_sha256=sha(up_bytes),source_verification_mode=source_mode,generated_utc=datetime.now(timezone.utc).isoformat(),files=files)
save_json(ROOT/'manifest.json',manifest)
out=WORK/'EDEN-02-RGBD-Acquisition-Results-v1.1.zip'
with zipfile.ZipFile(out,'w',compression=zipfile.ZIP_STORED,allowZip64=True) as export:
    for p in sorted(ROOT.rglob('*')):
        if p.is_file(): export.write(p,arcname='eden/'+str(p.relative_to(ROOT)))
with zipfile.ZipFile(out) as export:
    assert export.testzip() is None
    assert json.loads(export.read('eden/manifest.json'))['status']=='PASS_RGBD_PILOT'
if z is not None: z.close()
if reader is not None: reader.close()
print('PASS: EDEN-02 RGB-D pilot acquisition |',len(images),'pairs')
print('PENDING: manual montage review, depth unit and geometry verification')
print('Named output:',out,'| bytes:',out.stat().st_size)

## Next step — EDEN-03 spatial processing
Save Version → Save & Run All. Download the explicitly named results ZIP (or attach the saved output directly to EDEN-03). Inspect the montage; share the acquisition report, inventory and any failure output. EDEN-03 will segment vegetation and estimate canopy coverage, inspect depth conventions and calibration, and only then reconstruct visible-surface geometry. EDEN-01 remains the source of climate data.

**Troubleshooting:** HTTP 200/403/416 or range timeout: stop; use an authorized local archive/extracted canopy source. No pairs: inspect exact member names and supply reviewed override. Missing calibration: inspect config members and map camera IDs. Decode mismatch: do not resize to force pairing. Storage: lower pilot count; explicit named ZIP duplicates the pilot on disk. No GPU needed.

Source: Maree et al. (2025), [4th Autonomous Greenhouse Challenge: Dwarf Tomato Timeseries and Images, v1](https://doi.org/10.4121/fa102772-32db-4b30-bace-12f2016722ce.v1), CC BY 4.0. Camera mapping and local-time convention from official README captured in EDEN-00.